In [ ]:
import csv
import os
import pandas as pd

# 1. Configurazione dei file di input e di output finale
master_excel = "../../../../versions-3-spaces/datamodel/metadata-model-rockmechanics_V3_spaces.xlsx"
msl_excel = "../../../../versions-3-spaces/datamodel/msl-dictionaries.xlsx"
output_csv = "metadata-model-rockmechanics_V3_spaces_vocabularies.csv"

print("1. Estrazione in memoria dello sheet 'VOCABULARIES'...")
# Carichiamo la base strutturale dal tuo Excel Master
df_base = pd.read_excel(master_excel, sheet_name="VOCABULARIES", header=None)

# Trasformiamo il DataFrame in una lista di righe per manipolarla agevolmente
all_rows = df_base.values.tolist()
print(f"   -> Caricate {len(all_rows)} righe iniziali dal master.")

# 2. Caricamento globale dei fogli MSL
print(f"\n2. Lettura del file Excel MSL: '{msl_excel}'...")
all_msl_sheets = pd.read_excel(msl_excel, sheet_name=None)
print(f"   Rilevati {len(all_msl_sheets)} fogli MSL. Avvio unione verticale...")
print("-" * 80)

# 3. Costruzione della gigantesca griglia dati in memoria
for sheet_name, df in all_msl_sheets.items():
    clean_vocab_code = sheet_name.strip()
    
    # Inseriamo una riga vuota di stacco prima di agganciare il nuovo blocco vocabolario
    all_rows.append([])
    
    # Isolamento della descrizione globale del sotto-vocabolario
    try:
        global_description = df.iloc[1, 1] if not df.empty else f"Vocabulary for {clean_vocab_code}"
    except:
        global_description = f"Vocabulary for {clean_vocab_code}"
        
    # Pulizia righe vuote e isolamento dei record atomici
    df_data = df.iloc[3:].dropna(how="all") if len(df) > 3 else df.dropna(how="all")
    
    # --- INTESTAZIONE METADATI OPENBIS CON COLONNA VERSION = 5 ---
    all_rows.append(["VOCABULARY_TYPE", "", "", ""])
    all_rows.append(["Version", "Code", "Description", ""])
    all_rows.append(["5", clean_vocab_code, global_description, ""])
    all_rows.append(["Version", "Code", "Description", "Label"])
    
    # --- ACCUMULO DEI RECORD CON INIEZIONE VERSION = 5 ---
    for idx, (_, row) in enumerate(df_data.iterrows(), start=1):
        sequential_code = f"{clean_vocab_code}{idx}"
        
        # Estrazione di URI e Label dalle colonne
        uri_val = str(row.iloc[1]).strip() if len(row) > 1 else ""
        label_val = str(row.iloc[2]).strip() if len(row) > 2 else str(row.iloc[0]).strip()
        
        all_rows.append(["5", sequential_code, uri_val, label_val])
        
    print(f"   [OK] Dizionario '{clean_vocab_code}' accodato in memoria ({len(df_data)} lemmi).")

print("-" * 80)
print(f"3. Scrittura finale del file unico con codifica UTF-8-SIG (BOM Windows)...")

# 4. Scrittura fisica in modalità SOVRASCRITTURA ("w") con protezione dei caratteri speciali
with open(output_csv, mode="w", encoding="utf-8-sig", newline="") as f_out:
    writer = csv.writer(f_out, delimiter=";")
    
    for row in all_rows:
        # Pulizia estetica: convertiamo i valori nuli o NaN di Pandas in stringhe vuote ordinate
        clean_row = []
        for cell in row:
            if pd.isna(cell) or cell is None:
                clean_row.append("")
            else:
                clean_row.append(str(cell).strip())
        
        writer.writerow(clean_row)

print("=" * 80)
print(f"COMPLETATO CON SUCCESSO! Generato il file definitivo: '{output_csv}'")
print("I simboli ora si apriranno perfettamente sia su Excel Windows che sui server Linux di OpenBIS.")
print("=" * 80)

1. Estrazione in memoria dello sheet 'VOCABULARIES'...
   -> Caricate 431 righe iniziali dal master.

2. Lettura del file Excel MSL: '../../../../3/metadata-models/msl-dictionaries.xlsx'...
   Rilevati 17 fogli MSL. Avvio unione verticale...
--------------------------------------------------------------------------------
   [OK] Dizionario 'MSL_EQUIPMENT' accodato in memoria (331 lemmi).
   [OK] Dizionario 'MSL_MEASUREDPROPERTY' accodato in memoria (358 lemmi).
   [OK] Dizionario 'MSL_MINERAL' accodato in memoria (268 lemmi).
   [OK] Dizionario 'MSL_ARTIFICIAL' accodato in memoria (90 lemmi).
   [OK] Dizionario 'MSL_ROCK' accodato in memoria (204 lemmi).
   [OK] Dizionario 'MSL_SEDIMENT' accodato in memoria (22 lemmi).
   [OK] Dizionario 'MSL_POREFLUID' accodato in memoria (17 lemmi).
   [OK] Dizionario 'MSL_INFERRED' accodato in memoria (91 lemmi).
   [OK] Dizionario 'MSL_SETTING' accodato in memoria (191 lemmi).
   [OK] Dizionario 'MSL_MODEL' accodato in memoria (405 lemmi).
   [OK] 